In [26]:
#Load ios feature selected dataset for modeling:
import pandas as pd
ios_selected_features = pd.read_csv("../Processed_Data/Selected_FeaturesDatasets/ios_SelectedFeatures.csv")

**Train/Test Split**

In [27]:
#Preparing dataset for training:
#Select relevant columns for analysis:
y = ios_selected_features['stress']
X = ios_selected_features.drop(columns=['stress', 'uid', 'day'])  # Drop target, ID columns, and date
print("Final feature set columns:", X.columns)
print("Final feature set shape:", X.shape)  

Final feature set columns: Index(['Unnamed: 0.1', 'act_still_ep_0', 'race_more than one', 'Unnamed: 0',
       'audio_voice_ep_3', 'race_american indian/white', 'loc_self_dorm_still',
       'social_level', 'sleep_end', 'pam', 'loc_max_dis_from_campus_ep_0',
       'loc_study_convo_num', 'loc_study_unlock_duration', 'phq4_score',
       'unlock_num_ep_0', 'loc_self_dorm_audio_voice', 'act_walking_ep_0',
       'sse3-4', 'loc_study_dur', 'sse3-3',
       'race_american indian/alaska native', 'sse3_resp_mean', 'phq4-2',
       'gender', 'phq4-4', 'act_in_vehicle_ep_0', 'loc_self_dorm_dur',
       'race_alaskan native/white', 'race_white', 'avg_ema_spent_time',
       'act_still_ep_2', 'phq4-1', 'act_in_vehicle_ep_2', 'sleep_duration',
       'audio_convo_num_ep_3', 'quality_audio', 'loc_study_audio_voice',
       'quality_loc', 'loc_self_dorm_unlock_num', 'sse3-1',
       'loc_self_dorm_unlock_duration', 'sse3_resp_median', 'loc_study_still',
       'race_black', 'race_other/hispanic', '

In [28]:
#checking for missing values:
print("Missing values in each column:")
print(X.isnull().sum())

Missing values in each column:
Unnamed: 0.1                              0
act_still_ep_0                            0
race_more than one                        0
Unnamed: 0                                0
audio_voice_ep_3                          0
race_american indian/white                0
loc_self_dorm_still                   18650
social_level                              0
sleep_end                                 0
pam                                     374
loc_max_dis_from_campus_ep_0           5331
loc_study_convo_num                   19673
loc_study_unlock_duration             19673
phq4_score                                0
unlock_num_ep_0                           0
loc_self_dorm_audio_voice             18650
act_walking_ep_0                          0
sse3-4                                    0
loc_study_dur                            27
sse3-3                                    0
race_american indian/alaska native        0
sse3_resp_mean                            0
p

In [29]:
#Splitting data into test and train sets to prevent data leakage:
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold

#Column that identifies groups (participants):
group_col = 'uid'

#80/20 training testing split, with one testing group:
gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=ios_selected_features[group_col]))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = ios_selected_features[group_col].iloc[train_idx]

#Checking stress label distribution to ensure the groups are stratified:
print("Train distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest distribution:")
print(y_test.value_counts(normalize=True))

Train distribution:
stress
2.0    0.335620
3.0    0.292450
1.0    0.173687
4.0    0.144979
5.0    0.053264
Name: proportion, dtype: float64

Test distribution:
stress
2.0    0.359751
3.0    0.285915
1.0    0.171950
4.0    0.142055
5.0    0.040329
Name: proportion, dtype: float64


In [5]:

import numpy as np
y_pred_baseline = np.full_like(y_test, y_train.mean())

# 3. Evaluate baseline
from sklearn.metrics import mean_squared_error
baseline_mse = mean_squared_error(y_test, y_pred_baseline)

print("Baseline MSE:", baseline_mse)

Baseline MSE: 1.122066177499269


In [30]:
#Stratified group k-fold cross-validation to evaluate model performance:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(sgkf.split(X_train, y_train, groups=groups_train)):
    X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

**Random Forest Regressor - Global**

In [7]:
#Random forest regression model with stratifed group k-fold cross-validation:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score    


rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt', 
    random_state=42,
    n_jobs=-1)

RF_fold_mse = []
RF_fold_r2 = []

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X_train, y_train, groups=groups_train)):
    X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    rf_model.fit(X_fold_train, y_fold_train)
    val_preds = rf_model.predict(X_fold_val)
    
    mse = mean_squared_error(y_fold_val, val_preds)
    r2 = r2_score(y_fold_val, val_preds)
    RF_fold_mse.append(mse)
    RF_fold_r2.append(r2)
    print(f"Fold {fold+1} - MSE: {mse:.4f}, R^2: {r2:.4f}")

    
print(f"\nAverage MSE across folds: {sum(RF_fold_mse)/len(RF_fold_mse):.4f}")
print(f"Average R^2 across folds: {sum(RF_fold_r2)/len(RF_fold_r2):.4f}") 

Fold 1 - MSE: 0.7057, R^2: 0.4169
Fold 2 - MSE: 0.6817, R^2: 0.4295
Fold 3 - MSE: 0.7844, R^2: 0.3482
Fold 4 - MSE: 0.7589, R^2: 0.3699
Fold 5 - MSE: 0.7168, R^2: 0.4016

Average MSE across folds: 0.7295
Average R^2 across folds: 0.3932


In [8]:
#Final evaluation on the test set:
rf_model.fit(X_train, y_train)
test_preds = rf_model.predict(X_test)
test_mse = mean_squared_error(y_test, test_preds)
test_r2 = r2_score(y_test, test_preds)
print(f"\nTest Set - MSE: {test_mse:.4f}, R^2: {test_r2:.4f}")


Test Set - MSE: 0.7199, R^2: 0.3571


In [9]:
#Functions to convert regression predictions to class labels and compute accuracy:
from sklearn.metrics import accuracy_score
import numpy as np

# Define function to convert regression predictions to class labels based on binning:
def regression_to_class(y_pred):
    # Define bin edges
    bins = [1.5, 2.5, 3.5, 4.5]
    
    # Convert to class labels 1–5
    y_class = np.digitize(y_pred, bins) + 1
    
    return y_class

# Define function to compute per-class accuracy:
def per_class_accuracy(y_true, y_pred_class):
    classes = [1, 2, 3, 4, 5]
    class_acc = {}

    for c in classes:
        idx = (y_true == c)
        
        if np.sum(idx) == 0:
            class_acc[c] = None  # or "N/A"
        else:
            class_acc[c] = np.mean(y_pred_class[idx] == c)

    return class_acc

In [10]:
#Computing accuracy for by converting regression predictions to class labels:
# Regression predictions
y_pred_reg = rf_model.predict(X_test)

# Convert to classes
y_pred_reg = np.clip(y_pred_reg, 1, 5)
y_pred_class = regression_to_class(y_pred_reg)
y_test_int = y_test.astype(int)

# Accuracy
acc = accuracy_score(y_test_int, y_pred_class)

# Compute per-class accuracy
class_acc = per_class_accuracy(y_test_int, y_pred_class)

print(f"Regression → Classification Accuracy: {acc:.2f}")
print("\nPer-class accuracy:")
for c, acc in class_acc.items():
    if acc is None:
        print(f"Class {c}: N/A")
    else:
        print(f"Class {c}: {acc:.3f}")


Regression → Classification Accuracy: 0.44

Per-class accuracy:
Class 1: 0.086
Class 2: 0.632
Class 3: 0.622
Class 4: 0.106
Class 5: 0.010


**Linear Regression - Global**

In [35]:
import re

X_train = X_train.copy()

X_train.columns = [
    re.sub(r'[^A-Za-z0-9_]+', '_', col)
    for col in X_train.columns]

X_test = X_test.copy()
X_test.columns = [
    re.sub(r'[^A-Za-z0-9_]+', '_', col)
    for col in X_test.columns]

In [37]:
# LightGBM regression model with stratified group k-fold cross-validation:

from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, r2_score

lgbm_model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=10,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

LGBM_fold_mse = []
LGBM_fold_r2 = []

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X_train, y_train, groups=groups_train)):
    
    X_fold_train = X_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]
    y_fold_train = y_train.iloc[train_idx]
    y_fold_val = y_train.iloc[val_idx]
    
    lgbm_model.fit(X_fold_train, y_fold_train)
    
    val_preds = lgbm_model.predict(X_fold_val)
    
    mse = mean_squared_error(y_fold_val, val_preds)
    r2 = r2_score(y_fold_val, val_preds)
    
    LGBM_fold_mse.append(mse)
    LGBM_fold_r2.append(r2)
    
    print(f"Fold {fold+1} - MSE: {mse:.4f}, R^2: {r2:.4f}")

print(f"\nAverage MSE across folds: {sum(LGBM_fold_mse)/len(LGBM_fold_mse):.4f}")
print(f"Average R^2 across folds: {sum(LGBM_fold_r2)/len(LGBM_fold_r2):.4f}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001201 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6381
[LightGBM] [Info] Number of data points in the train set: 18304, number of used features: 45
[LightGBM] [Info] Start training from score 2.567035
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Fold 1 - MSE: 0.7030, R^2: 0.4192
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001204 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6366
[LightGBM] [Info] Number of data points in the train set: 18340, number of used features: 45
[LightGBM] [Info] Start training from score 2.571210
Fold

**XGBoost Regressor - Global**

In [11]:
#XGBoost regression model with stratifed group k-fold cross-validation:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

xgb_model = XGBRegressor(
    n_estimators=500,
    max_depth=10,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    random_state=42,
    n_jobs=-1)

XGB_fold_mse = []
XGB_fold_r2 = []

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X_train, y_train, groups=groups_train)):
    X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    xgb_model.fit(X_fold_train, y_fold_train)
    val_preds = xgb_model.predict(X_fold_val)
    
    mse = mean_squared_error(y_fold_val, val_preds)
    r2 = r2_score(y_fold_val, val_preds)
    XGB_fold_mse.append(mse)
    XGB_fold_r2.append(r2)
    print(f"Fold {fold+1} - MSE: {mse:.4f}, R^2: {r2:.4f}")

print(f"\nAverage MSE across folds: {sum(XGB_fold_mse)/len(XGB_fold_mse):.4f}")
print(f"Average R^2 across folds: {sum(XGB_fold_r2)/len(XGB_fold_r2):.4f}")

Fold 1 - MSE: 0.6881, R^2: 0.4315
Fold 2 - MSE: 0.6584, R^2: 0.4489
Fold 3 - MSE: 0.7687, R^2: 0.3612
Fold 4 - MSE: 0.7724, R^2: 0.3586
Fold 5 - MSE: 0.6915, R^2: 0.4227

Average MSE across folds: 0.7158
Average R^2 across folds: 0.4046


In [12]:
#Final evaluation on the test set:
xgb_model.fit(X_train, y_train)
test_preds = xgb_model.predict(X_test)
test_mse = mean_squared_error(y_test, test_preds)
test_r2 = r2_score(y_test, test_preds)
print(f"\nTest Set - MSE: {test_mse:.4f}, R^2: {test_r2:.4f}")


Test Set - MSE: 0.6820, R^2: 0.3909


In [13]:
#Computing accuracy for by converting regression predictions to class labels:
# Regression predictions
y_pred_reg = xgb_model.predict(X_test)

# Convert to classes
y_pred_reg = np.clip(y_pred_reg, 1, 5)
y_pred_class = regression_to_class(y_pred_reg)
y_test_int = y_test.astype(int)

# Accuracy
acc = accuracy_score(y_test_int, y_pred_class)

# Compute per-class accuracy
class_acc = per_class_accuracy(y_test_int, y_pred_class)

print(f"Regression → Classification Accuracy: {acc:.2f}")
print("\nPer-class accuracy:")
for c, acc in class_acc.items():
    if acc is None:
        print(f"Class {c}: N/A")
    else:
        print(f"Class {c}: {acc:.3f}")


Regression → Classification Accuracy: 0.46

Per-class accuracy:
Class 1: 0.197
Class 2: 0.636
Class 3: 0.604
Class 4: 0.179
Class 5: 0.045


**XGBoost Regressor - Personalized**

In [14]:
from sklearn.model_selection import train_test_split

#Training personalized models for each participant:
unique_participants = ios_selected_features['uid'].unique()

P_XGB_models = {}
P_XGB_train_metrics = {}

#Loop through each participant and train a personalized model:
for participant in unique_participants:
    
    participant_data = ios_selected_features[
        ios_selected_features['uid'] == participant
    ].sort_values('day') # Ensure data is sorted by day for time-based splitting
    
    X = participant_data.drop(columns=['stress', 'uid', 'day'])
    y = participant_data['stress']
    
    y = y.replace(5, 4)
    
    # Skip participants with too few data points for training:
    if len(participant_data) < 15:
        continue
    
    #Split into train/validation/test (60/20/20) with time-based splitting:
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.25, random_state=42
    )
    
    P_XGB_model = XGBRegressor(
        n_estimators=500,
        max_depth=10,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=3,
        random_state=42,
        n_jobs=-1)
    
    
    #Train with validation:
    P_XGB_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )
    
    train_preds = P_XGB_model.predict(X_train)
    val_preds = P_XGB_model.predict(X_val)
    
    train_mse = mean_squared_error(y_train, train_preds)
    val_mse = mean_squared_error(y_val, val_preds)
    
    P_XGB_train_metrics[participant] = {
        'train_mse': train_mse,
        'val_mse': val_mse
    }
    
    # Store model + test data for later
    P_XGB_models[participant] = {
        'model': P_XGB_model,
        'X_test': X_test,
        'y_test': y_test
    }

In [15]:
#Calculate and print average training and validation MSE across all personalized models:
avg_train_mse = sum(m['train_mse'] for m in P_XGB_train_metrics.values()) / len(P_XGB_train_metrics)
avg_val_mse = sum(m['val_mse'] for m in P_XGB_train_metrics.values()) / len(P_XGB_train_metrics)

print(f"Average TRAIN MSE: {avg_train_mse:.4f}")
print(f"Average VALIDATION MSE: {avg_val_mse:.4f}")

Average TRAIN MSE: 0.0084
Average VALIDATION MSE: 0.6218


In [16]:
#Calculate test metrics for each personalized model:
P_RF_test_metrics = {}

for participant, data in P_XGB_models.items():
    
    P_XGB_model = data['model']
    X_test = data['X_test']
    y_test = data['y_test']
    
    preds = P_XGB_model.predict(X_test)
    
    mse = mean_squared_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    P_RF_test_metrics[participant] = {
        'test_mse': mse,
        'test_r2': r2
    }

In [17]:
#Calculate and print average test metrics across all personalized models:
avg_test_mse = sum(m['test_mse'] for m in P_RF_test_metrics.values()) / len(P_RF_test_metrics)
avg_test_r2 = sum(m['test_r2'] for m in P_RF_test_metrics.values()) / len(P_RF_test_metrics)

print(f"\nAverage TEST MSE (FINAL): {avg_test_mse:.4f}")
print(f"Average TEST R^2 (FINAL): {avg_test_r2:.4f}")


Average TEST MSE (FINAL): 0.6080
Average TEST R^2 (FINAL): 0.1360


In [18]:
#Accuracy of personalized models:
y_pred_reg = P_XGB_model.predict(X_test)

# Convert to classes
y_pred_reg = np.clip(y_pred_reg, 1, 5)
y_pred_class = regression_to_class(y_pred_reg)
y_test_int = y_test.astype(int)

# Accuracy
acc = accuracy_score(y_test_int, y_pred_class)

# Compute per-class accuracy
class_acc = per_class_accuracy(y_test_int, y_pred_class)

print(f"Regression → Classification Accuracy: {acc:.2f}")
print("\nPer-class accuracy:")
for c, acc in class_acc.items():
    if acc is None:
        print(f"Class {c}: N/A")
    else:
        print(f"Class {c}: {acc:.3f}")


Regression → Classification Accuracy: 0.50

Per-class accuracy:
Class 1: 0.500
Class 2: 0.571
Class 3: 0.714
Class 4: 0.000
Class 5: N/A


**Random Forest Regressor - Personalized**

In [19]:
#Fallback function (only used if stratification fails)
def ensure_class_in_test(X, y, test_size=0.2):
    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)

    unique_classes = np.unique(y)
    test_indices = []

    # Ensure at least one sample per class in test
    for c in unique_classes:
        idx = np.where(y == c)[0]
        if len(idx) > 0:
            test_indices.append(idx[-1])

    remaining = list(set(range(len(y))) - set(test_indices))
    n_test = int(len(y) * test_size)

    if len(test_indices) < n_test and len(remaining) > 0:
        extra = np.random.choice(
            remaining,
            size=min(len(remaining), n_test - len(test_indices)),
            replace=False
        )
        test_indices.extend(extra)

    test_indices = np.array(test_indices)
    train_indices = np.array(list(set(range(len(y))) - set(test_indices)))

    return X.iloc[train_indices], X.iloc[test_indices], y.iloc[train_indices], y.iloc[test_indices]

In [20]:
P_RF_models = {}
P_RF_train_metrics = {}

# Loop through each participant
for participant in unique_participants:
    
    participant_data = ios_selected_features[
        ios_selected_features['uid'] == participant
    ].sort_values('day')  # keep time ordering
    
    X = participant_data.drop(columns=['stress', 'uid', 'day'])
    y = participant_data['stress']

    y = y.replace(5, 4)
    
    # Skip small datasets
    if len(participant_data) < 15:
        continue
    
    # Split (same as your setup)
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.25, random_state=42
    )
    
    #Random Forest model
    P_RF_model = RandomForestRegressor(
        n_estimators=500,
        max_depth=10,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features='sqrt', 
        random_state=42,
        n_jobs=-1)
    
    # Train (no eval_set here)
    P_RF_model.fit(X_train, y_train)
    
    # Predictions
    train_preds = P_RF_model.predict(X_train)
    val_preds = P_RF_model.predict(X_val)
    
    # Metrics
    train_mse = mean_squared_error(y_train, train_preds)
    val_mse = mean_squared_error(y_val, val_preds)
    
    P_RF_train_metrics[participant] = {
        'train_mse': train_mse,
        'val_mse': val_mse
    }
    
    # Store model + test data
    P_RF_models[participant] = {
        'model': P_RF_model,
        'X_test': X_test,
        'y_test': y_test
    }

In [21]:
#Calculate and print average training and validation MSE across all personalized models:
avg_train_mse = sum(m['train_mse'] for m in P_RF_train_metrics.values()) / len(P_RF_train_metrics)
avg_val_mse = sum(m['val_mse'] for m in P_RF_train_metrics.values()) / len(P_RF_train_metrics)

print(f"Average TRAIN MSE: {avg_train_mse:.4f}")
print(f"Average VALIDATION MSE: {avg_val_mse:.4f}")

Average TRAIN MSE: 0.3476
Average VALIDATION MSE: 0.5868


In [22]:
#Calculate test metrics for each personalized model:
P_RF_test_metrics = {}

for participant, data in P_RF_models.items():
    
    P_RF_model = data['model']
    X_test = data['X_test']
    y_test = data['y_test']
    
    preds = P_RF_model.predict(X_test)
    
    mse = mean_squared_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    P_RF_test_metrics[participant] = {
        'test_mse': mse,
        'test_r2': r2
    }

In [23]:
#Calculate and print average test metrics across all personalized models:
avg_test_mse = sum(m['test_mse'] for m in P_RF_test_metrics.values()) / len(P_RF_test_metrics)
avg_test_r2 = sum(m['test_r2'] for m in P_RF_test_metrics.values()) / len(P_RF_test_metrics)

print(f"\nAverage TEST MSE (FINAL): {avg_test_mse:.4f}")
print(f"Average TEST R^2 (FINAL): {avg_test_r2:.4f}")


Average TEST MSE (FINAL): 0.5881
Average TEST R^2 (FINAL): 0.1799


In [24]:
#Computing accuracy for by converting regression predictions to class labels:
# Regression predictions
y_pred_reg = P_RF_model.predict(X_test)

# Convert to classes
y_pred_reg = np.clip(y_pred_reg, 1, 5)
y_pred_class = regression_to_class(y_pred_reg)
y_test_int = y_test.astype(int)

# Accuracy
acc = accuracy_score(y_test_int, y_pred_class)

# Compute per-class accuracy
class_acc = per_class_accuracy(y_test_int, y_pred_class)

print(f"Regression → Classification Accuracy: {acc:.2f}")
print("\nPer-class accuracy:")
for c, acc in class_acc.items():
    if acc is None:
        print(f"Class {c}: N/A")
    else:
        print(f"Class {c}: {acc:.3f}")

Regression → Classification Accuracy: 0.55

Per-class accuracy:
Class 1: 0.000
Class 2: 0.857
Class 3: 0.714
Class 4: 0.000
Class 5: N/A
